In [ ]:
import gurobipy as gp
from gurobipy import GRB
import math
import pandas as pd
import ast

ROUNDS = [1, 2, 3]
venues_df = pd.read_csv("venues.csv")
VENUES = {
    row["venue"]: ast.literal_eval(row["coordinates"])
    for _, row in venues_df.iterrows()
}
timezone_df = pd.read_csv("timezones.csv")
TIMEZONES = dict(zip(timezone_df["city"], timezone_df["timezone"]))
df = pd.read_csv("optimized_groups.csv")
GROUPS = {
    row["group"]: [team.strip() for team in row["teams"].split(";")]
    for _, row in df.iterrows()
}
hosts_df = pd.read_csv("host_teams.csv")
HOST_VENUES = {
    row["host country"]: [v.strip() for v in row["host venues"].split(";")]
    for _, row in hosts_df.iterrows()
}
weights_df = pd.read_csv("fan_weights.csv")
FAN_WEIGHTS = dict(zip(weights_df["country"], weights_df["weight"]))

def haversine(coord1, coord2):
    lat1, lon1 = coord1
    lat2, lon2 = coord2
    R = 6371 # earth's radius in km
    phi1 = math.radians(lat1)
    phi2 = math.radians(lat2)
    delta_phi = math.radians(lat2 - lat1)
    delta_lambda = math.radians(lon2 - lon1)
    a = math.sin(delta_phi / 2) ** 2 + math.cos(phi1) * math.cos(phi2) * math.sin(delta_lambda / 2) ** 2
    c = 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))
    return R * c

def generate_matches(groups):
    matches = []
    for group, teams in groups.items():
        a, b, c, d = teams
        matches.extend([(group, a, b, 1), (group, c, d, 1),
                        (group, a, c, 2), (group, b, d, 2),
                        (group, a, d, 3), (group, b, c, 3)])
    return matches

def solve_travel_times(VENUES, TIMEZONES, GROUPS, ROUNDS, HOST_VENUES, TIMEZONE_WEIGHT=500, MAX_TRAVEL=4500, FAN_WEIGHT=0.1):
    model = gp.Model("optimize_travel_times")

    # need this to prevent the model from running for too long
    model.Params.TimeLimit = 1800
    model.Params.MIPGap = 0.01
    model.Params.MIPFocus = 1

    # create lists for venues, matches, match IDs, and teams
    venues = list(VENUES)
    matches = generate_matches(GROUPS)
    match_ids = range(len(matches))
    teams = [team for group in GROUPS.values() for team in group]
    venue_min = {v: 3 for v in venues}

    # map each team to the matches they play
    team_matches = {team: [] for team in teams}
    for m, (_, team1, team2, r) in enumerate(matches):
        team_matches[team1].append(m)
        team_matches[team2].append(m)

    # dictionaries to store distances and timezone differences between venues
    distances = {(v1, v2): haversine(VENUES[v1], VENUES[v2]) for v1 in venues for v2 in venues}
    timezone_diffs = {(v1, v2): abs(TIMEZONES[v1] - TIMEZONES[v2]) for v1 in venues for v2 in venues}

    # decision variables
    x = model.addVars(match_ids, venues, vtype=GRB.BINARY, name="x")
    z = model.addVars(teams, [1, 2], venues, venues, vtype=GRB.BINARY, name="z")

    # constraints

    # each host team muust play their first match in one of their host venues
    for team, host_venues in HOST_VENUES.items():
        m = team_matches[team][0]
        model.addConstr(gp.quicksum(x[m, v] for v in host_venues) == 1)

    # each match must be assigned to exactly one venue
    for m in match_ids:
        model.addConstr(gp.quicksum(x[m, v] for v in venues) == 1)

    # each team must play exactly one match each round
    for team in teams:
        for r in ROUNDS:
            model.addConstr(gp.quicksum(x[m, v] for m in team_matches[team] if matches[m][3] == r for v in venues) == 1)

    # each venue has to host at least 3 matches (variable to change though!)
    for v in venues:
        model.addConstr(gp.quicksum(x[m, v] for m in match_ids) >= venue_min[v])

    # link z to consecutive match venues and prohibit trips exceeding MAX_TRAVEL
    for team in teams:
        for r in [1, 2]:
            m1 = next(m for m in team_matches[team] if matches[m][3] == r)
            m2 = next(m for m in team_matches[team] if matches[m][3] == r + 1)
            for v1 in venues:
                for v2 in venues:
                    model.addConstr(z[team, r, v1, v2] <= x[m1, v1])
                    model.addConstr(z[team, r, v1, v2] <= x[m2, v2])
                    model.addConstr(z[team, r, v1, v2] >= x[m1, v1] + x[m2, v2] - 1)
                    if distances[v1, v2] > MAX_TRAVEL:
                        model.addConstr(z[team, r, v1, v2] == 0)

    # objective function minimizes team travel, timezone changes, and weighted fan travel
    travel_distance = gp.quicksum(distances[v1, v2] * z[team, r, v1, v2] for team in teams for r in [1, 2] for v1 in venues for v2 in venues)
    timezone_penalty = gp.quicksum(timezone_diffs[v1, v2] * z[team, r, v1, v2] for team in teams for r in [1, 2] for v1 in venues for v2 in venues)
    fan_travel = gp.quicksum(FAN_WEIGHTS[team] * distances[v1, v2] * z[team, r, v1, v2] for team in teams for r in [1, 2] for v1 in venues for v2 in venues)
    model.setObjective(travel_distance + FAN_WEIGHT * fan_travel +TIMEZONE_WEIGHT * timezone_penalty, GRB.MINIMIZE)
    model.optimize()

    if model.SolCount == 0:
        print("No feasible solution found.")
        return None

    schedule = {r: [] for r in ROUNDS}
    team_venues = {team: [None] * len(ROUNDS) for team in teams}

    # Track which match number each group is on
    group_match_count = {group: 0 for group in GROUPS}

    print("\n--- Optimized Match Schedule ---")
    print(f"{'Group':<8} {'Match ID':<10} {'Team 1':<22} {'Team 2':<22} {'City'}")

    schedule_rows = []

    for m in match_ids:
        group, team1, team2, r = matches[m]
        venue = next(v for v in venues if x[m, v].X > 0.5)

        group_match_count[group] += 1
        match_id = f"{group}_{group_match_count[group]}"

        schedule[r].append((group, team1, team2, venue))
        team_venues[team1][r - 1] = venue
        team_venues[team2][r - 1] = venue

        print(f"{group:<8} {match_id:<10} {team1:<22} {team2:<22} {venue}")

        schedule_rows.append({
            "Group": group,
            "Match ID": match_id,
            "Team 1": team1,
            "Team 2": team2,
            "City": venue
        })

    schedule_df = pd.DataFrame(schedule_rows)
    schedule_df = schedule_df.sort_values(["Group", "Match ID"])
    schedule_df.to_csv("optimized_venue_matching.csv", index=False)

    print("\n--- Optimized Team Venues ---")
    for team, venues_used in team_venues.items():
        print(f"{team}: {venues_used}")

    print(f"\nTotal Travel Distance: {travel_distance.getValue():,.0f} km")
    print(f"Total Timezone Changes: {timezone_penalty.getValue():.0f}")
    print(f"Objective Value: {model.ObjVal:,.0f}")

    return schedule, team_venues

solve_travel_times(VENUES, TIMEZONES, GROUPS, ROUNDS, HOST_VENUES, TIMEZONE_WEIGHT=500, MAX_TRAVEL=4500, FAN_WEIGHT=0.1)

Set parameter TimeLimit to value 1800
Set parameter MIPGap to value 0.01
Set parameter MIPFocus to value 1
Gurobi Optimizer version 13.0.2 build v13.0.2rc1 (win64 - Windows 11+.0 (26200.2))

CPU model: Intel(R) Core(TM) Ultra 9 185H, instruction set [SSE2|AVX|AVX2]
Thread count: 16 physical cores, 22 logical processors, using up to 22 threads

Non-default parameters:
TimeLimit  1800
MIPGap  0.01
MIPFocus  1

Optimize a model with 73963 rows, 25728 columns and 176656 nonzeros (Min)
Model fingerprint: 0x3a15a18f
Model has 23040 linear objective coefficients
Variable types: 0 continuous, 25728 integer (25728 binary)
Coefficient statistics:
  Matrix range     [1e+00, 1e+00]
  Objective range  [1e+02, 8e+03]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 3e+00]

Found heuristic solution: objective 293212.14686
Presolve removed 51796 rows and 2529 columns
Presolve time: 0.64s
Presolved: 22167 rows, 23199 columns, 68478 nonzeros
Variable types: 0 continuous, 23199 integer (23199

({1: [('A', 'Argentina', 'Mexico', 'Monterrey'),
   ('A', 'Norway', 'Congo DR', 'Monterrey'),
   ('B', 'Croatia', 'Austria', 'Toronto'),
   ('B', 'Canada', 'South Africa', 'Toronto'),
   ('C', 'Colombia', 'USA', 'San Francisco Bay Area'),
   ('C', 'Tunisia', 'Qatar', 'San Francisco Bay Area'),
   ('D', 'Spain', 'Uruguay', 'Seattle'),
   ('D', 'Uzbekistan', 'Haiti', 'Vancouver'),
   ('E', 'Portugal', 'Japan', 'Miami'),
   ('E', 'Algeria', 'Curaçao', 'Miami'),
   ('F', 'Brazil', 'IR Iran', 'New York/New Jersey'),
   ('F', 'Panama', 'Cabo Verde', 'Philadelphia'),
   ('G', 'Morocco', 'Switzerland', 'Atlanta'),
   ('G', 'Czechia', 'Jordan', 'Atlanta'),
   ('H', 'Netherlands', 'Korea Republic', 'Los Angeles'),
   ('H', 'Egypt', 'Bosnia & Herzegovina', 'Los Angeles'),
   ('I', 'England', 'Senegal', 'Dallas'),
   ('I', 'Sweden', 'New Zealand', 'Houston'),
   ('J', 'Germany', 'Ecuador', 'Kansas City'),
   ('J', "Côte d'Ivoire", 'Saudi Arabia', 'Kansas City'),
   ('K', 'France', 'Australia', 'Gu